# Control 2 - Solución
Nombre 1: [Tu Nombre]
Nombre 2: [Tu Nombre]

## Instrucciones
Este notebook contiene la solución completa del Control 2.

## Preliminar
**P0a) (1pt)** Cargue los archivos de la carpeta `./datasets` del repositorio en cinco diferentes dataframes.

In [ ]:
# Cargar librerías necesarias
library(dplyr)
library(lubridate)
library(ggplot2)
library(tidyr)

# Cargar los archivos (ajustar rutas según sea necesario)
cabecera <- read.csv("./datasets/cabecera.csv", stringsAsFactors = FALSE)
detalle <- read.csv("./datasets/detalle.csv", stringsAsFactors = FALSE)
clientes <- read.csv("./datasets/clientes.csv", stringsAsFactors = FALSE)
direcciones <- read.csv("./datasets/direcciones.csv", stringsAsFactors = FALSE)
producto <- read.csv("./datasets/producto.csv", stringsAsFactors = FALSE)

# Verificar que se cargaron correctamente
print("Archivos cargados correctamente:")
print(paste("cabecera:", nrow(cabecera), "filas"))
print(paste("detalle:", nrow(detalle), "filas"))
print(paste("clientes:", nrow(clientes), "filas"))
print(paste("direcciones:", nrow(direcciones), "filas"))
print(paste("producto:", nrow(producto), "filas"))

**P0b) (1pt)** Modifique el tipo de dato de los campos `fecha_del_pedido` y `fecha_de_envio` del dataframe _cabecera_ y conviértalos a tipo Date.

In [ ]:
# Convertir a tipo Date
cabecera$fecha_del_pedido <- as.Date(cabecera$fecha_del_pedido)
cabecera$fecha_de_envio <- as.Date(cabecera$fecha_de_envio)

# Verificar
str(cabecera[, c("fecha_del_pedido", "fecha_de_envio")])

## Preguntas 1.1: Validaciones iniciales

**P1a) (1pt)** Basándose en la tabla _cabecera_, ¿entre qué años se registran ventas?

In [ ]:
# P1a
years_range <- range(year(cabecera$fecha_del_pedido))
print(paste("Las ventas se registran entre los años", years_range[1], "y", years_range[2]))

**P1b) (2pt)** ¿Existe algún mes de algún año sin ventas?

In [ ]:
# P1b
all_months <- expand.grid(year = years_range[1]:years_range[2], month = 1:12)
months_with_sales <- unique(format(cabecera$fecha_del_pedido, "%Y-%m"))
months_missing <- setdiff(paste(all_months$year, all_months$month, sep = "-"), months_with_sales)

if(length(months_missing) == 0) {
  print("No existen meses sin ventas en el período analizado.")
} else {
  print(paste("Meses sin ventas:", paste(months_missing, collapse = ", ")))
}

**P1c) (2pt)** Valide que en la cabecera solo se registra una fila por `id_del_pedido`

In [ ]:
# P1c
duplicated_pedidos <- cabecera %>%
  group_by(id_del_pedido) %>%
  summarise(n = n()) %>%
  filter(n > 1)

if(nrow(duplicated_pedidos) == 0) {
  print("Validación exitosa: Cada pedido tiene una única fila en cabecera.")
} else {
  print(paste("ERROR: Existen", nrow(duplicated_pedidos), "pedidos duplicados en cabecera."))
}

**P1d) (2pt)** Valide que en el detalle de cada compra, todas las líneas de un mismo `id_del_pedido` se corresponden con un único `id_del_cliente`

In [ ]:
# P1d
clientes_por_pedido <- detalle %>%
  group_by(id_del_pedido) %>%
  summarise(n_clientes = n_distinct(id_del_cliente)) %>%
  filter(n_clientes > 1)

if(nrow(clientes_por_pedido) == 0) {
  print("Validación exitosa: Cada pedido está asociado a un único cliente en detalle.")
} else {
  print(paste("ERROR: Existen", nrow(clientes_por_pedido), "pedidos con múltiples clientes en detalle."))
}

**P1e) (1pt)** Corrobore que existen clientes con más de una dirección registrada en la tabla `direcciones`.

In [ ]:
# P1e
clientes_multiple_dir <- direcciones %>%
  group_by(id_del_cliente) %>%
  summarise(n_direcciones = n()) %>%
  filter(n_direcciones > 1)

print(paste("Existen", nrow(clientes_multiple_dir), "clientes con más de una dirección registrada."))
if(nrow(clientes_multiple_dir) > 0) {
  print(head(clientes_multiple_dir))
}

**P1f) (2pt)** Valide que en la tabla productos solo se registra una fila por `id_de_producto`

In [ ]:
# P1f
duplicated_productos <- producto %>%
  group_by(id_del_producto) %>%
  summarise(n = n()) %>%
  filter(n > 1)

if(nrow(duplicated_productos) == 0) {
  print("Validación exitosa: Cada producto tiene una única fila en la tabla productos.")
} else {
  print(paste("ERROR: Existen", nrow(duplicated_productos), "productos duplicados."))
}

## Preguntas 1.2

**P2a) (6pt)** ¿A qué países se han despachado productos durante el 2024 con método de envío "Mismo día"? Considere las ventas con `fecha_de_envio` en el 2024.

In [ ]:
# P2a
paises_mismo_dia_2024 <- cabecera %>%
  filter(year(fecha_de_envio) == 2024, metodo_de_envio == "Mismo día") %>%
  inner_join(detalle, by = "id_del_pedido") %>%
  inner_join(direcciones, by = "id_dir_cliente") %>%
  distinct(pais_region) %>%
  arrange(pais_region)

print("Países con envío 'Mismo día' en 2024:")
print(paises_mismo_dia_2024)

**P2b) (4pt)** De los países anteriores, ¿cuáles son los 5 con más envíos?

In [ ]:
# P2b
top_paises_mismo_dia <- cabecera %>%
  filter(year(fecha_de_envio) == 2024, metodo_de_envio == "Mismo día") %>%
  inner_join(detalle, by = "id_del_pedido") %>%
  inner_join(direcciones, by = "id_dir_cliente") %>%
  group_by(pais_region) %>%
  summarise(total_envios = n_distinct(id_del_pedido)) %>%
  arrange(desc(total_envios)) %>%
  head(5)

print("Top 5 países con más envíos 'Mismo día' en 2024:")
print(top_paises_mismo_dia)

## Preguntas 1.3

**P3a) (6pt)** Genere nuevas columnas en la tabla de detalle.

In [ ]:
# P3a - Agregar columnas calculadas
detalle <- detalle %>%
  mutate(
    venta_unitaria = ventas / cantidad,
    ganancia_unitaria = ganancia / cantidad,
    costo_unitario = venta_unitaria - ganancia_unitaria,
    precio_total = ventas / (1 - descuento),
    descuento_total_usd = precio_total - ventas,
    precio_unitario = precio_total / cantidad
  )

# Verificar primeras filas
head(detalle[, c("id_del_pedido", "ventas", "cantidad", "descuento", 
                  "venta_unitaria", "precio_unitario", "precio_total", "descuento_total_usd")])

**P3b) (5pt)** Listado de productos con precio más reciente y unidades vendidas.

In [ ]:
# P3b - Productos con precio más reciente y unidades vendidas
productos_resumen <- detalle %>%
  inner_join(cabecera, by = "id_del_pedido") %>%
  group_by(id_del_producto) %>%
  summarise(
    unidades_vendidas = sum(cantidad),
    fecha_mas_reciente = max(fecha_del_pedido),
    precio_unitario_actual = last(precio_unitario[order(fecha_del_pedido)])
  ) %>%
  inner_join(producto, by = "id_del_producto") %>%
  select(id_del_producto, precio_unitario_actual, unidades_vendidas, 
         nombre_del_producto, categoria, subcategoria) %>%
  arrange(desc(unidades_vendidas))

# Guardar como CSV
write.csv(productos_resumen, "productos_resumen.csv", row.names = FALSE)
print("Archivo 'productos_resumen.csv' creado exitosamente.")
print(paste("Total de productos:", nrow(productos_resumen)))
head(productos_resumen)

**P3c) (2pt)** ¿Cuáles son los 5 productos con más unidades vendidas?

In [ ]:
# P3c
top5_unidades <- productos_resumen %>%
  select(id_del_producto, nombre_del_producto, unidades_vendidas) %>%
  head(5)

print("Top 5 productos con más unidades vendidas:")
print(top5_unidades)

**P3d) (2pt)** Para los 5 productos anteriores, incluya descripción, categoría y subcategoría.

In [ ]:
# P3d (ya incluido en P3b, mostramos nuevamente)
print("Top 5 productos con detalles completos:")
print(productos_resumen %>% head(5))

**P3e) (4pt)** Producto con más unidades vendidas por año.

In [ ]:
# P3e
producto_top_anual <- detalle %>%
  inner_join(cabecera, by = "id_del_pedido") %>%
  mutate(anio = year(fecha_del_pedido)) %>%
  group_by(anio, id_del_producto) %>%
  summarise(unidades_vendidas = sum(cantidad)) %>%
  group_by(anio) %>%
  slice_max(order_by = unidades_vendidas, n = 1) %>%
  inner_join(producto, by = "id_del_producto") %>%
  select(anio, id_del_producto, nombre_del_producto, unidades_vendidas) %>%
  arrange(anio)

print("Producto más vendido por año:")
print(producto_top_anual)

# Verificar si cambia año a año
productos_cambian <- n_distinct(producto_top_anual$id_del_producto) > 1
if(productos_cambian) {
  print("El producto más vendido CAMBIA año a año.")
} else {
  print("El producto más vendido es el MISMO todos los años.")
}

## Preguntas 1.4: Análisis detallado de ventas

**P4a) (4pt)** Gráfico de líneas de ventas por año-mes.

In [ ]:
# P4a
ventas_mensuales <- detalle %>%
  inner_join(cabecera, by = "id_del_pedido") %>%
  mutate(anio_mes = format(fecha_del_pedido, "%Y-%m"),
         fecha_aux = as.Date(paste0(anio_mes, "-01"))) %>%
  group_by(fecha_aux) %>%
  summarise(ventas_totales = sum(ventas))

ggplot(ventas_mensuales, aes(x = fecha_aux, y = ventas_totales)) +
  geom_line(color = "steelblue", size = 1) +
  geom_smooth(method = "loess", se = FALSE, color = "red", linetype = "dashed") +
  labs(title = "Evolución de Ventas Mensuales",
       x = "Fecha (Año-Mes)",
       y = "Ventas Totales (USD)") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# Comentario sobre tendencia
print("Comentario: El gráfico muestra la evolución de ventas a lo largo del tiempo.")
print("La línea de tendencia (geom_smooth) permite visualizar patrones estacionales o tendencias generales.")

**P4b) (6pt)** Gráfico de barras de las 20 localidades con más ventas en 2024.

In [ ]:
# P4b
ventas_localidades_2024 <- cabecera %>%
  filter(year(fecha_de_envio) == 2024) %>%
  inner_join(detalle, by = "id_del_pedido") %>%
  inner_join(direcciones, by = "id_dir_cliente") %>%
  mutate(localidad = paste(pais_region, estado_provincia, sep = ", ")) %>%
  group_by(localidad) %>%
  summarise(ventas_totales = sum(ventas)) %>%
  arrange(desc(ventas_totales)) %>%
  head(20)

ggplot(ventas_localidades_2024, aes(x = reorder(localidad, ventas_totales), y = ventas_totales)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_text(aes(label = round(ventas_totales, 0)), hjust = -0.2, size = 3) +
  coord_flip() +
  labs(title = "Top 20 Localidades con Mayores Ventas en 2024",
       x = "Localidad (País, Estado/Provincia)",
       y = "Ventas Totales (USD)") +
  theme_minimal() +
  theme(axis.text.y = element_text(size = 8))